# Week 20: Advanced Computer Vision, 3D Vision & NeRFs
## Notebook 01: Object Detection: YOLO Mechanics, IoU & NMS from Scratch

### 1. Conceptual Intuition & Real-World Use Case
Image classification predicts *what* is in an image. Object detection predicts **what** and **where** (bounding box coordinates: $x, y, w, h$).
- Early detectors (R-CNN, Fast R-CNN) used two stages: candidate region proposals followed by classification (slow, 5-10 FPS).
- In 2016, Joseph Redmon et al. created **YOLO (You Only Look Once)**: frames object detection as a single unified regression problem directly from image pixels to bounding box coordinates in a single GPU pass (>60 FPS)!

---

### 2. Mathematical Foundations
1. **Intersection over Union (IoU):**
   $$\text{IoU}(A, B) = \frac{\text{Area}(A \cap B)}{\text{Area}(A \cup B)}$$
   - $\text{IoU} = 1.0$: Perfect overlap.
   - $\text{IoU} \ge 0.50$: Standard detection benchmark threshold.

2. **Non-Maximum Suppression (NMS):**
   Detectors propose hundreds of overlapping candidate boxes for a single object.
   - Sort candidate boxes by classification confidence.
   - Select the highest-confidence box and discard all remaining boxes with $\text{IoU} > \text{threshold}$ (typically $0.45$).
   - Repeat until all boxes are filtered.

In [ ]:
import numpy as np

# Step 1: Implement Intersection over Union (IoU) from Scratch
def compute_iou(box1, box2):
    """
    Boxes formatted as: [x1, y1, x2, y2] (top-left, bottom-right)
    """
    # Intersection coordinates
    x1_inter = max(box1[0], box2[0])
    y1_inter = max(box1[1], box2[1])
    x2_inter = min(box1[2], box2[2])
    y2_inter = min(box1[3], box2[3])
    
    inter_w = max(0.0, x2_inter - x1_inter)
    inter_h = max(0.0, y2_inter - y1_inter)
    intersection_area = inter_w * inter_h
    
    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
    
    union_area = box1_area + box2_area - intersection_area
    if union_area <= 0:
        return 0.0
    return intersection_area / union_area

# Step 2: Implement Non-Maximum Suppression (NMS)
def non_max_suppression(boxes, scores, iou_threshold=0.5):
    """
    boxes: array of shape (N, 4)
    scores: array of shape (N,)
    """
    # Sort by confidence score descending
    order = np.argsort(scores)[::-1]
    keep = []
    
    while len(order) > 0:
        best_idx = order[0]
        keep.append(best_idx)
        
        # Compute IoU of best box with remaining boxes
        ious = np.array([compute_iou(boxes[best_idx], boxes[i]) for i in order[1:]])
        
        # Keep boxes with IoU less than threshold
        remaining_indices = np.where(ious <= iou_threshold)[0]
        order = order[remaining_indices + 1]
        
    return keep

# Test Cases: Multiple overlapping candidate boxes detecting a single car
candidate_boxes = np.array([
    [100, 100, 210, 200],  # Box 0: High score
    [105, 102, 208, 198],  # Box 1: Near duplicate of Box 0
    [98,  104, 212, 205],  # Box 2: Duplicate
    [300, 300, 420, 450]   # Box 3: Completely different object (Pedestrian)
])
confidence_scores = np.array([0.92, 0.78, 0.85, 0.95])

kept_indices = non_max_suppression(candidate_boxes, confidence_scores, iou_threshold=0.45)

print("--- Object Detection NMS Results ---")
print(f"Original Candidate Boxes: {len(candidate_boxes)}")
print(f"Boxes Retained After NMS: {len(kept_indices)}")
for idx in kept_indices:
    print(f"  Box {idx}: Coordinates {candidate_boxes[idx]} | Score: {confidence_scores[idx]:.2f}")

### 3. Step-by-Step Code Walkthrough
1. **IoU Calculation:** Computes the overlapping geometric area divided by the total combined area.
2. **NMS Filtering:** Box 1 and Box 2 overlapped with Box 0 with $\text{IoU} > 0.80$, so NMS cleanly pruned them, preserving Box 0 and the independent Box 3.

---

### 4. Industry Applications & Trade-offs
- **Self-Driving Vehicles:** Real-time pedestrian and vehicle tracking at 60 FPS on edge hardware (NVIDIA Drive Orin).
- **Soft-NMS:** Rather than abruptly discarding overlapping boxes, Soft-NMS decays their scores exponentially, improving detection in crowded crowds.

---

### 5. Top Interview Questions & Answers
**Q1: What is the difference between Anchor-based and Anchor-free object detection?**
> **Answer:** Anchor-based models (YOLOv3/v4, Faster R-CNN) predefine fixed aspect-ratio template boxes and predict offsets. Anchor-free models (FCOS, CenterNet, YOLOv8) predict bounding box corners and center points directly, eliminating anchor hyperparameter tuning.

**Q2: What is CIoU (Complete IoU) Loss?**
> **Answer:** Standard IoU loss is zero when boxes do not overlap (zero gradient). CIoU accounts for three geometric factors: overlap area, normalized central point distance, and aspect ratio consistency, ensuring smooth gradient descent even when boxes are disjoint.